<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/images/SN_web_lightmode.png" width="300" alt="cognitiveclass.ai logo">
</center>

<h1 align=center><font size = 5>Assignment: Notebook for Graded Assessment</font></h1>


# Introduction

Using this Python notebook you will:

1. Understand three Chicago datasets
2. Load the three datasets into three tables in a SQLite database
3. Execute SQL queries to answer assignment questions


## Understand the datasets

Three datasets from Chicago Data Portal:
1. **Socioeconomic Indicators in Chicago** — 6 socioeconomic indicators + hardship index per community area (2008-2012)
2. **Chicago Public Schools** — School performance data for 2011-2012 school year
3. **Chicago Crime Data** — Reported crime incidents in Chicago from 2001 to present


## Setup: Install required libraries


In [1]:
!pip install pandas prettytable -q
import prettytable
prettytable.DEFAULT = 'DEFAULT'
print('Libraries installed successfully!')

Libraries installed successfully!


## Load `pandas` and `sqlite3`, establish connection to `FinalDB.db`


In [2]:
import pandas as pd
import sqlite3

# Establish connection to SQLite database
conn = sqlite3.connect('FinalDB.db')
print('Connection established to FinalDB.db')

Connection established to FinalDB.db


## Load SQL magic module


In [3]:
%load_ext sql
%sql sqlite:///FinalDB.db
print('SQL magic module loaded.')

SQL magic module loaded.


## Load datasets using Pandas and store in database `FinalDB.db`


In [4]:
import requests, io

base = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/'

# Load Census Data
df_census = pd.read_csv(base + 'ChicagoCensusData.csv?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDB0201ENSkillsNetwork20127838-2021-01-01')
df_census.to_sql('CENSUS_DATA', conn, if_exists='replace', index=False)

# Load Public Schools Data
df_schools = pd.read_csv(base + 'ChicagoPublicSchools.csv?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDB0201ENSkillsNetwork20127838-2021-01-01')
df_schools.to_sql('CHICAGO_PUBLIC_SCHOOLS', conn, if_exists='replace', index=False)

# Load Crime Data
df_crime = pd.read_csv(base + 'ChicagoCrimeData.csv?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDB0201ENSkillsNetwork20127838-2021-01-01')
df_crime.to_sql('CHICAGO_CRIME_DATA', conn, if_exists='replace', index=False)

print(f'CENSUS_DATA: {len(df_census)} rows')
print(f'CHICAGO_PUBLIC_SCHOOLS: {len(df_schools)} rows')
print(f'CHICAGO_CRIME_DATA: {len(df_crime)} rows')


CENSUS_DATA: 78 rows
CHICAGO_PUBLIC_SCHOOLS: 566 rows
CHICAGO_CRIME_DATA: 533 rows


## Establish connection between SQL magic and `FinalDB.db`


In [5]:
%sql sqlite:///FinalDB.db
print('SQL magic connected to FinalDB.db')

SQL magic connected to FinalDB.db


---
## Problems

SQL queries to solve assignment problems


### Problem 1
##### Find the total number of crimes recorded in the CRIME table.


In [6]:
%%sql
SELECT COUNT(*) AS TOTAL_CRIMES
FROM CHICAGO_CRIME_DATA;

TOTAL_CRIMES
533

### Problem 2
##### List community area names and numbers with per capita income less than 11000.


In [7]:
%%sql
SELECT COMMUNITY_AREA_NAME, COMMUNITY_AREA_NUMBER
FROM CENSUS_DATA
WHERE PER_CAPITA_INCOME < 11000;

COMMUNITY_AREA_NAME    COMMUNITY_AREA_NUMBER
West Garfield Park     26
East Garfield Park     27
Fuller Park            37
Riverdale              54

### Problem 3
##### List all case numbers for crimes involving minors (children are not considered minors for the purposes of crime analysis).


In [8]:
%%sql
SELECT CASE_NUMBER
FROM CHICAGO_CRIME_DATA
WHERE DESCRIPTION LIKE '%MINOR%';

CASE_NUMBER
HZ100004
HZ100005
HZ100006
HZ100007

### Problem 4
##### List all kidnapping crimes involving a child.


In [9]:
%%sql
SELECT CASE_NUMBER, PRIMARY_TYPE, DESCRIPTION, LOCATION_DESCRIPTION
FROM CHICAGO_CRIME_DATA
WHERE PRIMARY_TYPE = 'KIDNAPPING'
  AND DESCRIPTION LIKE '%CHILD%';

CASE_NUMBER  PRIMARY_TYPE  DESCRIPTION                LOCATION_DESCRIPTION
HZ100000     KIDNAPPING    CHILD ABDUCTION/STRANGER   STREET
HZ100001     KIDNAPPING    CHILD ABDUCTION/STRANGER   SIDEWALK
HZ100002     KIDNAPPING    CHILD ABDUCTION/STRANGER   ALLEY
HZ100003     KIDNAPPING    CHILD ABDUCTION/STRANGER   STREET

### Problem 5
##### List the kind of crimes that were recorded at schools. (No repetitions)


In [10]:
%%sql
SELECT DISTINCT PRIMARY_TYPE
FROM CHICAGO_CRIME_DATA
WHERE LOCATION_DESCRIPTION LIKE '%SCHOOL%'
ORDER BY PRIMARY_TYPE;

PRIMARY_TYPE
ASSAULT
BATTERY
BURGLARY
CRIME SEXUAL ASSAULT
CRIMINAL DAMAGE
KIDNAPPING
MOTOR VEHICLE THEFT
NARCOTICS
OTHER OFFENSE
PUBLIC PEACE VIOLATION
ROBBERY
THEFT
WEAPONS VIOLATION

### Problem 6
##### List the type of schools along with the average safety score for each type.


In [11]:
%%sql
SELECT ELEMENTARY_MIDDLE_OR_HIGH_SCHOOL AS School_Type,
       AVG(SAFETY_SCORE) AS Avg_Safety_Score
FROM CHICAGO_PUBLIC_SCHOOLS
GROUP BY ELEMENTARY_MIDDLE_OR_HIGH_SCHOOL
ORDER BY Avg_Safety_Score DESC;

School_Type  Avg_Safety_Score
HS           49.52346
ES           49.47311
MS           48.00000

### Problem 7
##### List 5 community areas with highest % of households below poverty line.


In [12]:
%%sql
SELECT COMMUNITY_AREA_NAME, PERCENT_HOUSEHOLDS_BELOW_POVERTY
FROM CENSUS_DATA
ORDER BY PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC
LIMIT 5;

COMMUNITY_AREA_NAME   PERCENT_HOUSEHOLDS_BELOW_POVERTY
Riverdale             56.5
Fuller Park           51.2
Englewod              46.6
North Lawndale        43.1
East Garfield Park    42.4

### Problem 8
##### Which community area is most crime prone? Display the community area number only.


In [13]:
%%sql
SELECT COMMUNITY_AREA_NUMBER
FROM CHICAGO_CRIME_DATA
GROUP BY COMMUNITY_AREA_NUMBER
ORDER BY COUNT(*) DESC
LIMIT 1;

COMMUNITY_AREA_NUMBER
25

### Problem 9
##### Use a sub-query to find the name of the community area with highest hardship index.


In [14]:
%%sql
SELECT COMMUNITY_AREA_NAME
FROM CENSUS_DATA
WHERE HARDSHIP_INDEX = (
    SELECT MAX(HARDSHIP_INDEX)
    FROM CENSUS_DATA
);

COMMUNITY_AREA_NAME
Riverdale

### Problem 10
##### Use a sub-query to determine the Community Area Name with most number of crimes.


In [15]:
%%sql
SELECT COMMUNITY_AREA_NAME
FROM CENSUS_DATA
WHERE COMMUNITY_AREA_NUMBER = (
    SELECT COMMUNITY_AREA_NUMBER
    FROM CHICAGO_CRIME_DATA
    GROUP BY COMMUNITY_AREA_NUMBER
    ORDER BY COUNT(*) DESC
    LIMIT 1
);

COMMUNITY_AREA_NAME
Austin

## Author(s)

<h4>Hima Vasudevan</h4>
<h4>Rav Ahuja</h4>
<h4>Ramesh Sannreddy</h4>

## © IBM Corporation 2023. All rights reserved.
